## Day 1: Delta Conversion & Optimization

### Architecture & Strategy
In a production Lakehouse architecture, raw data (Bronze layer) is often ingested as CSV or JSON. However, querying these raw formats directly is highly inefficient due to lack of indexing, schema enforcement, and transaction guarantees. 

Our strategy today focuses on foundational Data Engineering:
1. **Delta Conversion:** Upgrading raw CSVs to Delta format. Delta provides ACID transactions, scalable metadata handling, and time travel—all of which are critical prerequisites before building robust Machine Learning feature stores.
2. **Managed Tables:** Registering the data as a Managed Delta Table in Unity Catalog ensures Databricks manages both the metadata and the underlying storage lifecycle, simplifying governance.
3. **The "Small File Problem":** Streaming or frequent micro-batch ingestion often creates thousands of tiny files. This severely degrades I/O performance during ML training or analytical querying because the compute engine spends more time opening/closing files than reading data.
4. **Bin-Packing (OPTIMIZE):** We will use the `OPTIMIZE` command to actively compact these small files into larger, optimally sized files (typically ~1GB), drastically accelerating downstream read performance.

In [0]:
# Load November and October eCommerce CSVs from the established volume
nov_csv_path = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv"
oct_csv_path = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"

# Read both CSVs as DataFrames
nov_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(nov_csv_path)
oct_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(oct_csv_path)

# Combine both months into a single DataFrame
raw_df = nov_df.unionByName(oct_df)

# 1. Convert and Write to Delta format (Overwriting if it exists)
delta_storage_path = "/Volumes/workspace/ecommerce/ecommerce_data/events_bronze"

print("🚀 Writing combined CSV data to Delta format...")
raw_df.write.format("delta").mode("overwrite").save(delta_storage_path)

# Visualize the schema and a sample of the data
display(spark.read.format("delta").load(delta_storage_path).limit(5))

### Create a Managed Delta Table

In [0]:
# 2. Create a Managed Delta Table
# Using a managed table means Databricks (Unity Catalog) manages the lifecycle of the data. 
# We'll place it in our governed schema for production readiness.

catalog_name = "course_catalog"  # Fallback to 'hive_metastore' if on Community Edition
schema_name = "ecommerce_governed"
table_name = f"{catalog_name}.{schema_name}.events_delta_managed"

# Ensure schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

print(f"🛠️ Creating Managed Delta Table: {table_name}")

# Drop the table if it already exists to ensure idempotency in our pipeline
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Save the unified DataFrame (raw_df from Cell 1) directly as a managed table
raw_df.write.format("delta").mode("overwrite").saveAsTable(table_name)

# Verify the table exists and check its physical metadata
print("✅ Table created successfully. View storage details below:")
display(spark.sql(f"DESCRIBE DETAIL {table_name}"))

### Simulate the Small File Problem

In [0]:
import time

# 3. Append data multiple times to simulate the "Small File Problem"
print("⚠️ Simulating streaming/micro-batch ingestion (Creating small files)...")

# We will grab a tiny chunk of data and append it 10 times.
# In a real production environment, this happens when streaming jobs write data every few seconds/minutes.
df_micro_batch = spark.sql(f"SELECT * FROM {table_name} LIMIT 100")

for i in range(10):
    df_micro_batch.write.format("delta").mode("append").saveAsTable(table_name)
    
print("✅ Appends complete. Viewing Delta Transaction History:")

# Let's look at the transaction history to see all the separate write operations
# Notice how each append creates a new transaction (and new tiny parquet files)
display(
    spark.sql(f"DESCRIBE HISTORY {table_name}")
    .select("version", "timestamp", "operation", "operationParameters")
)

### Apply OPTIMIZE and Observe Improvement

In [0]:
# 4. Apply OPTIMIZE to compact the small files
print("🚀 Running OPTIMIZE to compact files (Bin-packing)...")

# Run the optimization command. This reads all the fragmented tiny files 
# and rewrites them into larger, optimal chunks (~1GB size).
optimize_results = spark.sql(f"OPTIMIZE {table_name}")

# Display the metrics of the optimization
# Notice the 'numFilesAdded' vs 'numFilesRemoved' columns. 
# You should see many small files removed and fewer large files added!
display(optimize_results)

# ---------------------------------------------------------
# 💡 PRO TIP (Commented out for future use):
# You can also use ZORDER here to colocate related information in the same set of files.
# If downstream ML models frequently filter by 'event_type' or 'user_id', ZORDER skips irrelevant data.
# spark.sql(f"OPTIMIZE {table_name} ZORDER BY (event_type)")
# ---------------------------------------------------------

### 🎯 Key Learnings & Interview Talking Points

If a recruiter asks about Data Engineering optimization or your Delta Lake experience, mention these concepts:
* **Delta vs. Parquet:** "While Parquet is a great columnar format, I exclusively use Delta Lake for production. Delta adds a transaction log over Parquet files, allowing for ACID transactions, concurrent reads/writes, and Time Travel—which is impossible with pure Parquet."
* **The Small File Problem:** "During micro-batching or streaming, Databricks generates thousands of tiny files. I know this creates a massive I/O bottleneck because the driver spends too much compute overhead opening and closing files during model training."
* **Bin-Packing & OPTIMIZE:** "To solve I/O bottlenecks, I implement automated `OPTIMIZE` jobs in my pipelines. This triggers a bin-packing algorithm that compacts small files into optimal ~1GB files, drastically reducing data scanning times for downstream ML tasks."
* **Managed vs. Unmanaged Tables:** "I utilized Managed Tables for this workflow to simplify data governance. By letting Unity Catalog and Databricks handle the underlying storage lifecycle, dropping a table automatically cleans up the storage, preventing orphaned data costs."